# CCP Literacy Estimation — Step-by-Step Demo (Model 1)

This notebook shows CCP estimation, transitions, CCS, a small MD fit, and a light EM loop.

In [1]:
import numpy as np, pandas as pd, json
from collections import defaultdict

CSV_PATH = 'simulated_panel.csv'
BETA=0.96
X_GRID = np.array([0.0,0.25,0.5,0.75,1.0])
J=len(X_GRID)
A_BINS,Y_BINS,AGE_BINS=6,4,4
ALPHA=5.0
H,DRAWS=3,30
K_TYPES=2
EULER_GAMMA=0.5772156649015329

## 1) Load the panel data

**Why**: We need a person–time panel with $(a,y,age,x)$ to build states and choices.


In [2]:
df = pd.read_csv(CSV_PATH)
df.head()

,id,t,a,y,x,age,s,c,type
0,1,0,51882.159995,25000.0,0.00,30.0,0.80,15583.960639,2
1,1,1,62335.842556,25000.0,0.25,31.0,0.80,14179.793847,2
2,1,2,56719.175390,25000.0,1.00,32.0,0.80,11954.547584,2
3,1,3,47818.190334,25000.0,0.25,33.0,0.60,23211.704425,2
4,1,4,34817.556638,25000.0,0.25,34.0,0.95,2255.859542,2


## 2) Define state bins and discretize actions

- CCPs and transitions are nonparametric objects. To estimate them robustly, we need a finite state grid. Quantile bins give balanced cells.
- A discrete action set $J=\{0,0.25,0.5,0.75,1\}$ makes logit CCPs tractable and aligns with Hotz–Miller.

In [3]:
def quantile_bins(series, nbins):
    qs = np.linspace(0,1,nbins+1)
    cuts = series.quantile(qs).values.astype(float)
    for i in range(1,len(cuts)):
        if cuts[i] <= cuts[i-1]:
            cuts[i] = cuts[i-1] + 1e-9
    return cuts

def cut_to_bins(x, cuts):
    return int(np.clip(np.searchsorted(cuts, x, side='right')-1, 0, len(cuts)-2))

def dirichlet_smooth(counts, alpha=5.0):
    counts = np.asarray(counts,float)
    prior = alpha/len(counts)
    return (counts + prior)/(counts.sum()+alpha)

def inclusive_value_from_ccp(P):
    P = np.clip(np.asarray(P,float), 1e-12,1-1e-12)
    return -float(np.log(P[0]))

def softmax(v):
    v = np.asarray(v,float); v = v - np.max(v)
    ex = np.exp(v); return ex/ex.sum()

In [4]:
a_cuts = quantile_bins(df['a'], A_BINS)
y_cuts = quantile_bins(df['y'], Y_BINS)
age_cuts = quantile_bins(df['age'], AGE_BINS)

dfb = df.copy()
dfb['ia'] = dfb['a'].apply(lambda v: cut_to_bins(v, a_cuts))
dfb['iy'] = dfb['y'].apply(lambda v: cut_to_bins(v, y_cuts))
dfb['iage'] = dfb['age'].apply(lambda v: cut_to_bins(v, age_cuts))

def x_to_j(x): 
    return int(np.argmin(np.abs(X_GRID - float(x))))

dfb['j'] = dfb['x'].apply(x_to_j)

dfb = dfb.sort_values(['id','t']).reset_index(drop=True)

for col in ['ia','iy','iage','j']:
    dfb[col+'_next'] = dfb.groupby('id')[col].shift(-1)

dfb = dfb.dropna(subset=['ia_next','iy_next','iage_next']).copy()

for col in ['ia_next','iy_next','iage_next','j_next']:
    dfb[col] = dfb[col].astype(int)

dfb.head()

,id,t,a,y,x,age,s,c,type,ia,iy,iage,j,ia_next,iy_next,iage_next,j_next
0,1,0,51882.159995,25000.0,0.00,30.0,0.80,15583.960639,2,3,0,0,0,4,0,0,1
1,1,1,62335.842556,25000.0,0.25,31.0,0.80,14179.793847,2,4,0,0,1,4,0,1,4
2,1,2,56719.175390,25000.0,1.00,32.0,0.80,11954.547584,2,4,0,1,4,3,0,1,1
3,1,3,47818.190334,25000.0,0.25,33.0,0.60,23211.704425,2,3,0,1,1,1,0,2,1
4,1,4,34817.556638,25000.0,0.25,34.0,0.95,2255.859542,2,1,0,2,1,2,0,2,3


## 3) Estimate CCP $\hat{P}(j \mid s)$

With logit shocks, CCPs invert to value-index differences and yield an inclusive value for the state. They are the data bridge to dynamic values.

In [272]:
J=len(X_GRID)
counts = defaultdict(lambda: np.zeros(J, float))

for _,row in dfb.iterrows():
    s=(int(row['ia']), int(row['iy']), int(row['iage']))
    j=int(row['j'])
    counts[s][j]+=1.0

ccp={}; state_counts={}

for s,cvec in counts.items():
    state_counts[s]=float(cvec.sum())
    ccp[s]=dirichlet_smooth(cvec, alpha=ALPHA)
len(ccp)

24

In [273]:
counts

defaultdict(<function __main__.<lambda>()>,
            {(3, 0, 0): array([29., 33., 30., 22., 34.]),
             (4, 0, 0): array([36., 36., 39., 38., 47.]),
             (4, 0, 1): array([36., 47., 23., 40., 40.]),
             (3, 0, 1): array([46., 38., 41., 41., 41.]),
             (1, 0, 2): array([39., 41., 46., 47., 35.]),
             (2, 0, 2): array([44., 52., 39., 53., 37.]),
             (3, 0, 3): array([27., 17., 30., 25., 21.]),
             (5, 0, 0): array([34., 53., 45., 49., 53.]),
             (5, 0, 1): array([45., 34., 44., 28., 44.]),
             (5, 0, 2): array([31., 37., 45., 40., 39.]),
             (5, 0, 3): array([22., 14., 12., 18., 16.]),
             (4, 0, 2): array([40., 45., 45., 33., 48.]),
             (4, 0, 3): array([24., 18., 23., 24., 23.]),
             (0, 0, 0): array([65., 67., 67., 72., 72.]),
             (1, 0, 1): array([34., 52., 52., 41., 42.]),
             (2, 0, 1): array([38., 52., 33., 34., 50.]),
             (2, 0, 0): arra

In [274]:
ccp

{(3,
  0,
  0): array([0.19607843, 0.22222222, 0.20261438, 0.1503268 , 0.22875817]),
 (4,
  0,
  0): array([0.1840796 , 0.1840796 , 0.19900498, 0.19402985, 0.23880597]),
 (4,
  0,
  1): array([0.19371728, 0.2513089 , 0.12565445, 0.21465969, 0.21465969]),
 (3,
  0,
  1): array([0.22169811, 0.18396226, 0.19811321, 0.19811321, 0.19811321]),
 (1,
  0,
  2): array([0.18779343, 0.1971831 , 0.22065728, 0.22535211, 0.16901408]),
 (2,
  0,
  2): array([0.19565217, 0.23043478, 0.17391304, 0.23478261, 0.16521739]),
 (3, 0, 3): array([0.224, 0.144, 0.248, 0.208, 0.176]),
 (5,
  0,
  0): array([0.14644351, 0.22594142, 0.19246862, 0.20920502, 0.22594142]),
 (5, 0, 1): array([0.23 , 0.175, 0.225, 0.145, 0.225]),
 (5,
  0,
  2): array([0.16243655, 0.1928934 , 0.23350254, 0.20812183, 0.20304569]),
 (5,
  0,
  3): array([0.26436782, 0.17241379, 0.14942529, 0.2183908 , 0.1954023 ]),
 (4,
  0,
  2): array([0.18981481, 0.21296296, 0.21296296, 0.15740741, 0.22685185]),
 (4,
  0,
  3): array([0.21367521, 0.1

## 4) Estimate transitions $\hat{p}(s' \mid s,j)$

To compute the continuation under “what if I choos $j$ now?”, we must know how states move given current state+action. We estimate it empirically from the panel.

In [275]:
trans = defaultdict(lambda: defaultdict(float))

for _, row in dfb.iterrows():
    s=(int(row['ia']), int(row['iy']), int(row['iage']))
    j=int(row['j'])
    sp=(int(row['ia_next']), int(row['iy_next']), int(row['iage_next']))
    trans[(s,j)][sp]+=1.0

trans_prob={}
for key,d in trans.items():
    items=list(d.items())
    probs=np.array([v for (_,v) in items], float); probs=probs/probs.sum()
    trans_prob[key]=([sp for (sp,_) in items], probs)

def draw_next_state(s,j):
    key=(s,j)
    if key not in trans_prob:
        cand=[(k,v) for (k,v) in trans_prob.items() if k[0]==s]
        if not cand:
            return s
        sps=[]; ps=[]
        for (_, (sp_list, p_list)) in cand:
            sps+=sp_list; ps+=list(p_list/len(cand))
        ps=np.array(ps,float); ps=ps/ps.sum()
        idx=np.random.choice(len(sps), p=ps)
        return sps[idx]
    sps,ps=trans_prob[key]
    idx=np.random.choice(len(sps), p=ps)
    return sps[idx]

In [276]:
trans_prob

{((3, 0, 0),
  0): ([(4, 0, 0),
   (5, 0, 0),
   (2, 0, 1),
   (5, 0, 1),
   (4, 0, 1),
   (3, 0, 0),
   (2, 0, 0),
   (3,
    0,
    1)], array([0.31034483, 0.20689655, 0.13793103, 0.13793103, 0.10344828,
         0.03448276, 0.03448276, 0.03448276])),
 ((4, 0, 0),
  1): ([(4, 0, 1),
   (5, 0, 1),
   (3, 0, 1),
   (4, 0, 0),
   (2, 0, 1),
   (3, 0, 0),
   (2, 0, 0),
   (5,
    0,
    0)], array([0.27777778, 0.13888889, 0.08333333, 0.19444444, 0.08333333,
         0.05555556, 0.11111111, 0.05555556])),
 ((4, 0, 1),
  4): ([(3, 0, 1),
   (2, 0, 2),
   (1, 0, 2),
   (4, 0, 2),
   (1, 0, 1),
   (5, 0, 2),
   (3, 0, 2),
   (5, 0, 1),
   (2, 0, 1),
   (4,
    0,
    1)], array([0.1  , 0.1  , 0.075, 0.15 , 0.1  , 0.05 , 0.075, 0.075, 0.125,
         0.15 ])),
 ((3, 0, 1),
  1): ([(1, 0, 2),
   (3, 0, 1),
   (4, 0, 2),
   (3, 0, 2),
   (1, 0, 1),
   (0, 0, 1),
   (4, 0, 1),
   (2,
    0,
    1)], array([0.18421053, 0.13157895, 0.21052632, 0.18421053, 0.13157895,
         0.02631579, 0.1052631

## 5) Compute inclusive values and CCS $\Delta EV (s ; j,0)$

Under logit, the $\textbf{ex-ante value}$ at a state is
$$V(s) = \gamma_E - \ln P(0 \mid s)$$
So we can $\textbf{simulate forward}$ using $\hat{P}$ and $\hat{p}$, summing discounted $\textbf{inclusive values}$, instead of solving Bellman.


In [277]:
states=list(ccp.keys())

def delta_ev_ccs(s,j):
    def path_val(start_s, initial_j):
        vals=[]
        for _ in range(DRAWS):
            s_current=draw_next_state(start_s, initial_j)
            acc=(BETA**1)*inclusive_value_from_ccp(ccp.get(s_current, np.ones(J)/J))
            for h in range(2,H+1):
                P_current=ccp.get(s_current, None)
                if P_current is None:
                    j_draw=0
                else:
                    j_draw=int(np.random.choice(J, p=P_current))
                s_current=draw_next_state(s_current, j_draw)
                acc+=(BETA**h)*inclusive_value_from_ccp(ccp.get(s_current, np.ones(J)/J))
            vals.append(acc)
        return float(np.mean(vals)) if vals else 0.0
    return path_val(s,j) - path_val(s,0)

delta_cache={}
for s in states:
    for j in range(1,J):
        delta_cache[(s,j)] = delta_ev_ccs(s,j)

len(delta_cache)

96

In [278]:
delta_cache

{((3, 0, 0), 1): -0.043460835355764615,
 ((3, 0, 0), 2): -0.019165669951242137,
 ((3, 0, 0), 3): -0.07760684320885414,
 ((3, 0, 0), 4): -0.05098050060588921,
 ((4, 0, 0), 1): -0.07334522985148251,
 ((4, 0, 0), 2): -0.19684043421785447,
 ((4, 0, 0), 3): -0.07043311907061778,
 ((4, 0, 0), 4): -0.1693905374888045,
 ((4, 0, 1), 1): -0.004307097541615512,
 ((4, 0, 1), 2): 0.11430835897070057,
 ((4, 0, 1), 3): 0.021095902086298857,
 ((4, 0, 1), 4): 0.12295829026950322,
 ((3, 0, 1), 1): 0.09009326847233812,
 ((3, 0, 1), 2): 0.17566395751295172,
 ((3, 0, 1), 3): 0.10765610587310714,
 ((3, 0, 1), 4): -0.029875380864122114,
 ((1, 0, 2), 1): 0.2901843347544997,
 ((1, 0, 2), 2): -0.05414312817212341,
 ((1, 0, 2), 3): 0.32242521424374093,
 ((1, 0, 2), 4): 0.13958487072592796,
 ((2, 0, 2), 1): 0.09630019484766894,
 ((2, 0, 2), 2): 0.0667556901071169,
 ((2, 0, 2), 3): 0.15993054679578922,
 ((2, 0, 2), 4): 0.03358498679285571,
 ((3, 0, 3), 1): 0.21431700078838078,
 ((3, 0, 3), 2): 0.21233985047055004,

## 6) Minimum-distance (MD) fit of cost parameters

The HM moment says \textbf{empirical log-odds} must equal $(- \text{expected cost}) + \beta \times \text{CCS difference}.$

With latent types $L$, expected cost is $\textbf{mixture-weighted}$: $\sum_L \pi_L \big(k_0 + k_L + \varphi_L a \big).$


In [279]:
a_mids = np.array([0.5*(a_cuts[b]+a_cuts[b+1]) for b in range(len(a_cuts)-1)], float)

def objective(theta):
    k0, kL_low, kL_high, phi_low, phi_high, pi_low = theta
    pi=np.array([pi_low, max(1e-6, 1.0-pi_low)]); pi=pi/np.sum(pi)
    loss=0.0; wsum=0.0
    for s in states:
        ia,iy,iage = s
        P = ccp[s]
        a_mid = a_mids[ia]
        costs = np.array([k0 + kL_low + phi_low*a_mid, k0 + kL_high + phi_high*a_mid])
        exp_cost = float(np.sum(pi * costs))
        for j in range(1,J):
            logodds = float(np.log(P[j]) - np.log(P[0]))
            lam = -exp_cost + BETA * delta_cache[(s,j)]
            w = max(state_counts.get(s,1.0), 1.0)
            loss += (logodds - lam)**2 * w
            wsum += w
    return loss / max(wsum,1.0)

theta0 = np.array([20.0, 20.0, 5.0, 0.01, 0.005, 0.5], float)
rng = np.random.default_rng(0)
best_theta = theta0.copy(); best_val = objective(best_theta)
for it in range(200):
    cand = best_theta + rng.normal(0, [2,2,2,0.001,0.001,0.05], size=6)
    val = objective(cand)
    if val < best_val:
        best_theta, best_val = cand, val
best_val, best_theta

(141.49322126033553,
 array([ 6.49109223e+00,  3.30331823e+01,  3.33099195e+00, -7.69926920e-04,
         1.06379242e-04,  5.57136395e-01]))

## 7) EM: infer laten literacy (personal level)

The HM moment says \textbf{empirical log-odds} must equal $(- \text{expected cost}) + \beta \times \text{CCS difference}.$

With latent types $L$, expected cost is $\textbf{mixture-weighted}$: $\sum_L \pi_L \big(k_0 + k_L + \varphi_L a \big).$


In [280]:
def type_ccp(theta):
    k0, kL_low, kL_high, phi_low, phi_high, pi_low = theta
    def ccps_at_state(s, L):
        ia,_,_ = s
        a_mid = a_mids[ia]
        if L==0:
            kL, phi = kL_low, phi_low
        else:
            kL, phi = kL_high, phi_high
        lam = np.zeros(J); lam[0]=0.0
        for j in range(1,J):
            lam[j] = - (k0 + kL + phi*a_mid) + BETA * delta_cache[(s,j)]
        return softmax(lam)
    return ccps_at_state

traj = defaultdict(list)
for _, row in dfb.sort_values(['id','t']).iterrows():
    s_t = (int(row['ia']), int(row['iy']), int(row['iage']))
    j_t = int(row['j'])
    traj[int(row['id'])].append((s_t, j_t))

def em(theta_init, n_iter=2):
    theta = theta_init.copy()
    k0, kL_low, kL_high, phi_low, phi_high, pi_low = theta
    pi = np.array([pi_low, max(1e-6, 1.0-pi_low)]); pi = pi/np.sum(pi)
    for it in range(n_iter):
        ccps = type_ccp(theta)
        omega={}
        for pid, seq in traj.items():
            like=np.zeros(2)
            for L in [0,1]:
                ll=0.0
                for (s_t, j_t) in seq:
                    P = ccps(s_t, L)
                    ll += np.log(max(P[j_t], 1e-12))
                like[L]=np.exp(ll)
            post = pi * like
            s = post.sum()
            omega[pid] = post/s if s>0 else np.array([0.5,0.5])
        # update pi
        mat = np.stack(list(omega.values()))
        pi = mat.mean(axis=0); pi = pi/np.sum(pi)
        # small local search to re-fit costs keeping pi fixed
        def objective_fixed_pi(th):
            th = th.copy()
            th[-1] = pi[0]
            return objective(th)
        best = theta.copy(); best[-1] = pi[0]
        best_val = objective(best)
        for _ in range(100):
            cand = best + np.random.normal(0, [1,1,1,0.0005,0.0005,0.0], size=6)
            val = objective_fixed_pi(cand)
            if val < best_val:
                best, best_val = cand, val
        theta = best; theta[-1] = pi[0]
    return theta, pi, omega

theta_em, pi_em, omega = em(best_theta, n_iter=2)
theta_em, pi_em

(array([-6.31965809e+00,  3.18973951e+01,  3.75729348e+00,  3.14992746e-03,
        -1.20846153e-04,  4.00000001e-02]),
 array([0.04, 0.96]))

In [281]:
rows=[]
for pid, post in omega.items():
    rows.append({'id':pid, 'p_low': float(post[0]), 'p_high': float(post[1])})
post_df = pd.DataFrame(rows).sort_values('id').reset_index(drop=True)
post_df.head()

,id,p_low,p_high
0,1,8.498416e-57,1.0
1,2,4.145641e-80,1.0
2,3,7.557323e-10,1.0
3,4,1.011179e-56,1.0
4,5,4.764394e-80,1.0


In [282]:
post_df

,id,p_low,p_high
0,1,8.498416e-57,1.0
1,2,4.145641e-80,1.0
2,3,7.557323e-10,1.0
3,4,1.011179e-56,1.0
4,5,4.764394e-80,1.0
...,...,...,...
595,596,3.176742e-33,1.0
596,597,1.721093e-56,1.0
597,598,1.507663e-56,1.0
598,599,1.512853e-56,1.0


In [283]:
summary = {
    'theta_em': list(map(float, theta_em)),
    'pi_em': list(map(float, pi_em)),
    'objective_value': float(objective(theta_em))
}
print(json.dumps(summary, indent=2))

{
  "theta_em": [
    -6.319658092355856,
    31.89739513018396,
    3.757293477564619,
    0.00314992746326427,
    -0.00012084615289409864,
    0.040000000081243
  ],
  "pi_em": [
    0.040000000081243,
    0.9599999999187571
  ],
  "objective_value": 1.1079708024843702
}
